# 2.4 — Do raw às janelas e features (Forma 2, CSV)

Aqui está o **ponto forte da Forma 2**: como guardamos o sinal bruto, nós mesmos definimos a
janela no Colab — podemos **descartar a transição** e recortar o trecho exato do fenômeno
antes de janelar (a Forma 1 já chegou janelada do edge, sem essa liberdade).

Geramos as **7 features do edge** (`mean_ax/ay/az`, `std_ax/ay/az`, `rms_mag`) **mais 5 novas**
sobre a magnitude do sinal (`std_mag`, `p2p_mag`, `crest_mag`, `kurt_mag`, `zcr_mag`) por
**janela fixa de 100 amostras (1 s) sem sobreposição**, e salvamos um CSV de features para o
notebook **2.5**. As 12 ficam todas no CSV de propósito — é o notebook 2.5 que vai comparar os
dois grupos e mostrar por que nem toda feature que "funciona" é uma feature boa.

> **Atende Sprint 4 – item 1:** features por janela, dataset balanceado (≥30 por classe).

In [ ]:
!pip install -q pandas numpy

## Carregar o raw

In [ ]:
import pandas as pd, numpy as np, re

try:
    from google.colab import files
    enviados = files.upload()
    arquivos = list(enviados.keys())
except Exception:
    import glob
    arquivos = glob.glob("coleta_*.csv")


def extrai_rodada(nome_arquivo):
    # Nome novo do coletor_raw.py: coleta_<label>_<NN>_<AAAAMMDD>_<HHMMSS>.csv
    # O label pode ter underscore (ex.: "inclinado_frente"), entao NAO damos
    # split() ingenuo — ancoramos no sufixo fixo de data/hora e pegamos o NN
    # logo antes dele. CSVs antigos (sem NN) caem no fallback "01": uma unica
    # rodada implicita.
    m = re.search(r"_(\d+)_\d{8}_\d{6}\.csv$", nome_arquivo)
    return m.group(1) if m else "01"


partes = []
for a in arquivos:
    g = pd.read_csv(a)
    g["rodada"] = extrai_rodada(a)
    partes.append(g)

raw = pd.concat(partes, ignore_index=True)
raw["timestamp"] = pd.to_datetime(raw["timestamp"])
print(raw.groupby(["label", "rodada"]).size())

## Checar a taxa de amostragem efetiva (por rodada)

O CSV já traz o timestamp de recepção — até agora ninguém o usava. Estimamos a fs real de cada
rodada a partir dele: é o sanity check que revela problema de timing no firmware ou latência de
Serial/USB **antes** de treinar qualquer coisa. Esperado ≈ 100 Hz (o `AMOSTRA_MS` do
`app17-10`); desvio grande é bandeira vermelha, não estatística de features.

In [ ]:
FS_NOMINAL = 100  # Hz, o AMOSTRA_MS do app17-10

def fs_efetiva(g):
    dt = g.sort_values("timestamp")["timestamp"].diff().dt.total_seconds().dropna()
    return 1 / dt.median()

print("fs efetiva por rodada (esperado ~100 Hz):")
for (lab, rod), g in raw.groupby(["label", "rodada"]):
    fs = fs_efetiva(g)
    desvio = abs(fs - FS_NOMINAL) / FS_NOMINAL
    alerta = "  <<< desvio > 5%, investigar antes de treinar" if desvio > 0.05 else ""
    print(f"  {lab:20s} rodada {rod}: fs={fs:6.1f} Hz{alerta}")

## Descartar a transição (Aula 14, slide 22)

Os primeiros instantes de cada rodada costumam ter ruído (mão ajustando/reposicionando o
sensor). Descartamos as primeiras `DESCARTE_S` segundos **de cada rodada** — não da classe
inteira. Ajuste à vontade — essa flexibilidade é exatamente o que a Forma 2 oferece.

> **Correção importante:** descartar por `groupby("label")` (sem a rodada) cortaria a
> transição só da *primeira* rodada concatenada de cada classe, deixando a transição
> contaminada em todas as outras. Por isso agrupamos por `["label", "rodada"]`.

In [ ]:
FS = 100               # Hz
DESCARTE_S = 3         # segundos descartados no inicio de CADA RODADA

partes = []
for (lab, rod), g in raw.groupby(["label", "rodada"]):
    g = g.sort_values("timestamp").iloc[DESCARTE_S * FS:]
    partes.append(g)
raw_limpo = pd.concat(partes, ignore_index=True)
print("Amostras após descarte:")
print(raw_limpo.groupby(["label", "rodada"]).size())

## Janela fixa (100 amostras, sem sobreposição) → 12 features

As fórmulas de `std`/`rms` usam divisão por N (`np.std` com `ddof=0`), que bate com o
`calcStd` do ESP32. As 5 novas features são calculadas sobre a **magnitude**
`mag = sqrt(ax²+ay²+az²)`, e a maioria sobre a componente **AC** (`mag - mean(mag)`, ou seja,
sem o offset de ~9,81 da gravidade):

| feature | o que mede |
|---|---|
| `std_mag` | RMS da vibração "pura" (sem o bias da gravidade) |
| `p2p_mag` | amplitude pico-a-pico da magnitude |
| `crest_mag` | fator de crista — sensível a impacto/impulso |
| `kurt_mag` | curtose — indicador clássico de falha impulsiva (rolamento) |
| `zcr_mag` | taxa de cruzamento por zero — proxy barato de frequência, sem FFT |

In [ ]:
TAMANHO_JANELA = 100   # 100 amostras @ 100 Hz = 1 s

def features_da_janela(j):
    ax, ay, az = j["ax"].values, j["ay"].values, j["az"].values
    mag = np.sqrt(ax**2 + ay**2 + az**2)
    mag_ac = mag - mag.mean()   # componente AC: remove o offset da gravidade
    std_mag = mag_ac.std()
    zcr = np.mean(np.diff(np.sign(mag_ac)) != 0)

    return {
        # --- as 7 originais do edge (mantidas de proposito: o notebook 2.5
        #     compara com as novas e mostra por que elas enganam aqui) ---
        "mean_ax": ax.mean(), "mean_ay": ay.mean(), "mean_az": az.mean(),
        "std_ax":  ax.std(),  "std_ay":  ay.std(),  "std_az":  az.std(),
        "rms_mag": np.sqrt(np.mean(mag**2)),
        # --- novas, sobre a magnitude (item 3 do plano) ---
        "std_mag":   std_mag,
        "p2p_mag":   mag.max() - mag.min(),
        "crest_mag": (np.max(np.abs(mag_ac)) / std_mag) if std_mag > 0 else 0.0,
        "kurt_mag":  pd.Series(mag_ac).kurtosis(),
        "zcr_mag":   zcr,
    }

linhas = []
for (lab, rod), g in raw_limpo.groupby(["label", "rodada"]):
    g = g.sort_values("timestamp").reset_index(drop=True)
    n_janelas = len(g) // TAMANHO_JANELA
    for k in range(n_janelas):
        janela = g.iloc[k*TAMANHO_JANELA:(k+1)*TAMANHO_JANELA]
        feat = features_da_janela(janela)
        feat["label"] = lab
        feat["rodada"] = rod
        feat["ordem"] = k          # posicao temporal da janela DENTRO da rodada
        linhas.append(feat)

feat_df = pd.DataFrame(linhas)
print("Janelas por classe:")
print(feat_df["label"].value_counts())
assert (feat_df["label"].value_counts() >= 30).all(), "Menos de 30 janelas em alguma classe — colete mais!"
feat_df.head()

## Split treino/teste SEM vazamento

Janelas vizinhas no tempo são quase iguais — um split aleatório (ou cronológico dentro de uma
única rodada) vaza informação (Aula 14, slide 25). Preferimos o **split por rodada**: a última
rodada de cada classe vai inteira para teste, o resto para treino — assim o modelo é avaliado
numa sessão de coleta que ele nunca viu, o teste mais realista de generalização.

Se uma classe só tiver 1 rodada, caímos de volta no split cronológico 70/30 dentro dela — ainda
funciona, mas é mais fraco (janelas vizinhas da mesma rodada se parecem). Colete pelo menos
2 rodadas por classe para escapar desse fallback; o notebook 2.5 usa a coluna `rodada` para
`LeaveOneGroupOut`, que precisa de várias.

In [ ]:
feat_df["split"] = ""
for lab, g in feat_df.groupby("label"):
    rodadas = sorted(g["rodada"].unique())
    if len(rodadas) >= 2:
        rodada_teste = rodadas[-1]  # ultima rodada da classe -> teste (holdout)
        feat_df.loc[g.index[g["rodada"] == rodada_teste], "split"] = "teste"
        feat_df.loc[g.index[g["rodada"] != rodada_teste], "split"] = "treino"
    else:
        # so 1 rodada nesta classe: fallback cronologico 70/30 (mais fraco).
        ordenado = g.sort_values("ordem")
        corte = int(len(ordenado) * 0.7)
        feat_df.loc[ordenado.index[:corte],  "split"] = "treino"
        feat_df.loc[ordenado.index[corte:], "split"] = "teste"

feat_df.to_csv("features_from_raw.csv", index=False)
print(feat_df.groupby(["label", "rodada", "split"]).size())
try:
    from google.colab import files
    files.download("features_from_raw.csv")
except Exception:
    print("Salvo em features_from_raw.csv")

## Comparativo raw × features (entrega da Sprint 4)

- **Raw:** 1 linha = 1 leitura (~6000/classe/min). Útil para visualizar e re-janelar.
- **Features:** 1 linha = 1 janela de 1 s, 12 colunas (as 7 originais do edge + as 5 novas
  sobre a magnitude). É o que vai para o modelo.

Leve o `features_from_raw.csv` para o notebook **2.5**: lá comparamos as 7 originais contra as
novas e mostramos por que `mean_ax/ay/az` e `rms_mag` são um atalho enganoso *neste* problema
binário (normal × anômalo). Guarde essa conclusão — no notebook **2.6** (5 classes do motor,
`app17-11`), o mesmo `mean_*` vira a família de features que resolve as classes de inclinação.
Mesma feature, dois veredictos opostos, dependendo da pergunta.